# IndicatorsEnv v4.0 — Colab Test Notebook

Test the multi-stock relative-alpha MDP environment and run inference with a small HF model.

**What this notebook does:**
1. Upload & extract the hackathon code
2. Install dependencies
3. Start the IndicatorsEnv server (FastAPI on port 7860)
4. Smoke test — verify multi-step structure, grader, and baseline
5. Full inference — run `inference.py` with a 1B model against the live environment

**IndicatorsEnv v4.0 — Multi-stock Relative Alpha MDP:**
- Agent observes 3 stocks from the same NSE sector at each step
- Picks one stock (or passes with NONE) and declares Bullish/Bearish
- Reward = (chosen stock return − sector average) × direction × conviction × 50
- Market-neutral: random policy earns ~0 expected reward
- Step spacing: short=1 day (5 steps), medium=5 days (10 steps), long=20 days (15 steps)
- Task 3: early termination if drawdown > 10%, macro context in observation

## Step 1: Upload & Extract Code

Create the zip on your Mac first, then upload it here:

```bash
# Run this in your terminal (Mac)
cd "/Users/aaryanmanawat/Aaryan/StockAnalyzer Pro/version3.0/StockAnalyzer-Pro"
zip -r hackathon.zip hackathon/env hackathon/inference.py hackathon/requirements.txt
```

Then run the cell below and select `hackathon.zip` when the file picker appears.

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()  # Select hackathon.zip from your Mac

with zipfile.ZipFile('hackathon.zip', 'r') as z:
    z.extractall('/content/')

%cd /content/hackathon
print(f'Ready in: {os.getcwd()}')
print('Files:', os.listdir('.'))

## Step 2: Install Dependencies

In [ ]:
!pip install -q fastapi uvicorn yfinance pandas numpy requests openai
print('Done')

## Step 3: Start the IndicatorsEnv Server

Starts FastAPI on `http://localhost:7860`.
The first `reset()` call takes ~20–40s as yfinance fetches 3 stocks' OHLCV data.

> **Note:** Uses `workers=1` — required because session state is in-memory.

In [ ]:
import subprocess, time, sys, requests

server = subprocess.Popen(
    [sys.executable, 'indicators_env.py'],
    cwd='/content/hackathon/env',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
time.sleep(12)

try:
    resp = requests.get('http://localhost:7860/health', timeout=5)
    h = resp.json()
    print('Server:', h)
    print('Version:', h.get('version'))
    print('Episode steps:', h.get('episode_steps', {}))
    print('Step spacing:', h.get('step_spacing', {}))
    print('Sectors:', h.get('sectors', []))
except Exception as e:
    print('Not ready:', e)
    out = server.stdout.read1(2048).decode('utf-8', errors='replace')
    print('Startup log:', out)

## Step 4: Smoke Test (No LLM Required)

Verifies five things:
- **Multi-stock observation**: reset returns `available_stocks` (3 symbols), `stocks` dict, `sector`
- **Multi-step structure**: `done=False` on steps 1–4, `done=True` on step 5 (short task)
- **Relative alpha reward**: reward based on chosen stock vs sector average
- **NONE action**: passing a step gives reward=0 and is not counted in grader
- **Grader**: returns `score > 0` on active (non-NONE) steps
- **Baseline**: short task gets ≥50 steps (5 steps × 10 episodes), score ≈ 0.33

In [ ]:
import requests

BASE = 'http://localhost:7860'

# ── Reset (short task = 5 steps, 1-day spacing) ───────────────────────────────
r = requests.post(f'{BASE}/reset', params={'term': 'short'}, timeout=90)
assert r.status_code == 200, f'Reset failed: {r.text}'
data       = r.json()
session_id = data['info']['session_id']
obs        = data['observation']
sector     = obs['sector']
available  = obs['available_stocks']
print(f'Reset OK — sector={sector}  stocks={available}')
print(f'Step={obs["step"]}/{obs["max_steps"]}  term={obs["term"]}')
print(f'Stocks in obs: {list(obs["stocks"].keys())}')
print(f'Session: {session_id}\n')

# ── 5 Steps ────────────────────────────────────────────────────────────────────
print('Steps (alternating Bullish pick / NONE pass):')
for i in range(1, 6):
    # Alternate: pick first stock on odd steps, NONE on even steps
    if i % 2 == 1:
        chosen_stock = available[0]
        direction    = 'Bullish'
        conviction   = 0.7
    else:
        chosen_stock = 'NONE'
        direction    = 'NONE'
        conviction   = 0.0

    s = requests.post(
        f'{BASE}/step',
        params={'session_id': session_id},
        json={'stock': chosen_stock, 'direction': direction, 'conviction': conviction},
        timeout=30,
    ).json()
    step    = s['info']['step']
    done    = s['done']
    reward  = s['reward']
    alpha   = s['info'].get('alpha_pct', 'N/A')
    gt      = s['info'].get('ground_truth', 'N/A')
    ok      = (i < 5 and not done) or (i == 5 and done)
    print(f'  step={step}  done={done}  action={chosen_stock}/{direction}  '
          f'reward={reward:.4f}  alpha={alpha}%  GT={gt}  {"✅" if ok else "❌ WRONG"}')

# ── Grader — active steps only (steps 1, 3, 5 = Bullish picks) ───────────────
print()
g = requests.post(f'{BASE}/grader', json={
    'task_id': 'short_term_direction',
    'episode_results': [
        {'predicted': 'Bullish', 'ground_truth': 'Bullish', 'conviction': 0.7},
        {'predicted': 'NONE',    'ground_truth': 'N/A',     'conviction': 0.0},
        {'predicted': 'Bearish', 'ground_truth': 'Bearish', 'conviction': 0.8},
        {'predicted': 'NONE',    'ground_truth': 'N/A',     'conviction': 0.0},
        {'predicted': 'Bullish', 'ground_truth': 'Neutral',  'conviction': 0.5},
    ],
}).json()
score  = g['score']
active = g['breakdown'].get('active_steps', '?')
print(f'Grader: score={score}  active_steps={active}  {"✅" if score > 0 else "❌ WRONG"}')

# ── Baseline (takes ~60–120s with real yfinance) ──────────────────────────────
print('\nRunning baseline — this takes ~60–120s...')
b = requests.get(f'{BASE}/baseline', timeout=600).json()
short_eps = b['tasks']['short_term_direction']['num_episodes']
mean      = b['overall_mean']
ok        = short_eps >= 50   # 5 steps × 10 episodes (may be more if baseline loops)
print(f'short_term  num_episodes={short_eps}  {"✅" if ok else f"❌ WRONG (expected ≥50)"}')
print(f'medium_term num_episodes={b["tasks"]["medium_term_direction"]["num_episodes"]}  (expect ≥100)')
print(f'long_term   num_episodes={b["tasks"]["long_term_conviction"]["num_episodes"]}  (expect ≤150)')
print(f'overall_mean={mean}  (expect ~0.33 for random agent)')

## Step 5: Full Inference with a 1B Model

Runs `inference.py` end-to-end against the live environment.
Uses `meta-llama/Llama-3.2-1B-Instruct` via the HF Inference API (free tier).

**Paste your HF token below.** `--n_episodes 1` = 30 total LLM calls (5+10+15 steps).

> Get your token at: https://huggingface.co/settings/tokens (read access is enough)

In [ ]:
import os

# ── Paste your HF token below ─────────────────────────────────────────────────
HF_TOKEN = 'hf_YOUR_TOKEN_HERE'   # ← replace this
# ─────────────────────────────────────────────────────────────────────────────

assert HF_TOKEN != 'hf_YOUR_TOKEN_HERE', 'Replace HF_TOKEN with your real token first!'

os.environ['API_BASE_URL'] = 'https://router.huggingface.co/v1/'
os.environ['MODEL_NAME']   = 'meta-llama/Llama-3.2-1B-Instruct'
os.environ['HF_TOKEN']     = HF_TOKEN

!python /content/hackathon/inference.py \
    --env_url http://localhost:7860 \
    --n_episodes 1

### Expected Output

```
[START] task=short_term_direction env=IndicatorsEnv model=meta-llama/Llama-3.2-1B-Instruct
[STEP]  step=1 stock=HDFCBANK action=Bullish reward=0.3200 done=False error=None
[STEP]  step=2 stock=NONE     action=NONE    reward=0.0000 done=False error=None
[STEP]  step=3 stock=ICICIBANK action=Bearish reward=-0.1500 done=False error=None
[STEP]  step=4 stock=HDFCBANK  action=Bullish reward=0.2800 done=False error=None
[STEP]  step=5 stock=AXISBANK  action=Bullish reward=0.4100 done=True  error=None
[END]   success=True steps=5 score=0.XXXX rewards=[...]

[START] task=medium_term_direction ...
... (10 [STEP] lines) ...
[END]   success=True steps=10 ...

[START] task=long_term_conviction ...
... (up to 15 [STEP] lines, may end early if drawdown > 10%) ...
[END]   success=True steps=≤15 ...
```

**What to check:**
- `done=False` on steps 1–4, `done=True` on step 5 for short task
- `steps=5` for short, `steps=10` for medium, `steps≤15` for long
- `score > 0` in every `[END]` line
- Rewards are floats (alpha × direction × conviction × 50, may be 0.0 for NONE steps)
- 3 tasks × 5+10+15 = 30 total `[STEP]` lines per `n_episodes=1` run
- Some steps may show `stock=NONE action=NONE` — this is correct selective participation